# Data Preparation

In [2]:
!nvidia-smi 

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
%pip install -r requirements.txt

  Using cached Sastrawi-1.0.1-py2.py3-none-any.whl.metadata (909 bytes)
  Using cached better_profanity-0.7.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached torch-2.14.0-cp311-cp311-win_amd64.whl.metadata (38 kB)
  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
  Using cached filelock-4.0.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
Using cached Sastrawi-1.0.1-py2.py3-none-any.whl (209 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import torch
if torch.cuda.is_available():
  print("CUDA available. GPU will be used for computation.")
  device = 0 # Default to the first GPU; adjust if you have multiple GPUs
else:
  print("CUDA not available. CPU will be used for computation.")
  device = -1 # Indicates CPU usage

CUDA not available. CPU will be used for computation.


In [8]:
import numpy as np
import pandas as pd
import re

In [9]:
# 1. Load Data
df = pd.read_csv("News.csv")
df.head()

,id,id_author,title,portal,time,author,editor,content,source
0,0,1,Infografis Pekerja Asing Dilarang Masuk Wilaya...,Liputan6.com,"24 Jul 2021, 09:02 WIB",Abdillah,Abdillah,Pemerintah melalui Menteri Hukum dan Hak Asasi...,https://www.liputan6.com/news/read/4614451/inf...
1,1,1,Infografis Jadwal Bulu Tangkis Indonesia di Ol...,Liputan6.com,"23 Jul 2021, 23:23 WIB",Abdillah,Abdillah,Bulu Tangkis menjadi andalan Indonesia berburu...,https://www.liputan6.com/bola/read/4614427/inf...
2,2,1,"Infografis Jangan Bebal, Kamu Tidak Kebal Covi...",Liputan6.com,"23 Jul 2021, 10:40 WIB",Abdillah,Abdillah,Covid-19 tidak mengenal usia dan status. Siapa...,https://www.liputan6.com/news/read/4613233/inf...
3,3,1,Infografis Awas Perokok Lebih Rentan Tertular ...,Liputan6.com,"22 Jul 2021, 10:35 WIB",Abdillah,Abdillah,Kebiasaan merokok berisiko menimbulkan sejumla...,https://www.liputan6.com/news/read/4612324/inf...
4,4,1,Infografis Perbedaan Aturan PPKM Level 3 dan 4,Liputan6.com,"22 Jul 2021, 09:01 WIB",Abdillah,Abdillah,Pemberlakuan Pembatasan Kegiatan Masyarakat at...,https://www.liputan6.com/news/read/4612511/inf...


In [10]:
# 2. Clean data
df = df.dropna(subset=["content"])  # drop rows where 'content' is missing

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"<.*?>", " ", text)               # remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)       # remove punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()          # collapse multiple spaces
    return text

df["cleaned_content"] = df["content"].apply(clean_text)
df.head(3)

,id,id_author,title,portal,time,author,editor,content,source,cleaned_content
0,0,1,Infografis Pekerja Asing Dilarang Masuk Wilaya...,Liputan6.com,"24 Jul 2021, 09:02 WIB",Abdillah,Abdillah,Pemerintah melalui Menteri Hukum dan Hak Asasi...,https://www.liputan6.com/news/read/4614451/inf...,Pemerintah melalui Menteri Hukum dan Hak Asasi...
1,1,1,Infografis Jadwal Bulu Tangkis Indonesia di Ol...,Liputan6.com,"23 Jul 2021, 23:23 WIB",Abdillah,Abdillah,Bulu Tangkis menjadi andalan Indonesia berburu...,https://www.liputan6.com/bola/read/4614427/inf...,Bulu Tangkis menjadi andalan Indonesia berburu...
2,2,1,"Infografis Jangan Bebal, Kamu Tidak Kebal Covi...",Liputan6.com,"23 Jul 2021, 10:40 WIB",Abdillah,Abdillah,Covid-19 tidak mengenal usia dan status. Siapa...,https://www.liputan6.com/news/read/4613233/inf...,Covid 19 tidak mengenal usia dan status Siapa ...


In [11]:
# 3 & 4. Tokenization + Lowercasing
def tokenize(text):
    text = text.lower()          # lowercasing
    tokens = text.split()        # simple whitespace tokenization
    return tokens

df["tokens"] = df["cleaned_content"].apply(tokenize)
df[["content", "tokens"]].head(3)

,content,tokens
0,Pemerintah melalui Menteri Hukum dan Hak Asasi...,"[pemerintah, melalui, menteri, hukum, dan, hak..."
1,Bulu Tangkis menjadi andalan Indonesia berburu...,"[bulu, tangkis, menjadi, andalan, indonesia, b..."
2,Covid-19 tidak mengenal usia dan status. Siapa...,"[covid, 19, tidak, mengenal, usia, dan, status..."


In [12]:
# 5. Build Vocabulary of Unique Terms
from collections import Counter

# Flatten all tokens across all documents into one big list
all_tokens = [token for tokens in df["tokens"] for token in tokens]

# Count frequency of each term across the corpus (useful for inspection, not just vocab)
term_freq = Counter(all_tokens)

# Build vocabulary: sorted list of unique terms
vocab = sorted(term_freq.keys())

# Map each term to an index (needed later for tf/tf-idf vector construction)
vocab_to_idx = {term: idx for idx, term in enumerate(vocab)}

print(f"Total tokens in corpus: {len(all_tokens)}")
print(f"Vocabulary size (unique terms): {len(vocab)}")
print("Sample vocab terms:", vocab[:10])

Total tokens in corpus: 5690989
Vocabulary size (unique terms): 90462
Sample vocab terms: ['0', '00', '000', '0000', '0001', '00010', '000100', '0001000', '0002', '00023']


In [13]:
# Most common terms
most_common_df = pd.DataFrame(term_freq.most_common(15), columns=["term", "frequency"])

# Least common terms
least_common_df = pd.DataFrame(list(term_freq.items())[-15:], columns=["term", "frequency"])

print("Most common terms:")
display(most_common_df)

print("Least common terms (sample):")
display(least_common_df)

Most common terms:


,term,frequency
0,yang,131750
1,dan,115197
2,di,101584
3,untuk,57911
4,ini,57851
5,dengan,52194
6,dari,48863
7,itu,41400
8,dalam,39489
9,pada,32856


Least common terms (sample):


,term,frequency
0,komcad,23
1,psdn,3
2,kesukarelaan,1
3,bsnp,1
4,mengonversinya,1
5,hirarki,1
6,militeristik,3
7,tjetjep,4
8,kesarjanaan,2
9,berpatok,1


In [14]:
# Create 5 queries
# Check docs include candidate topics
candidate_keywords = ["covid", "vaksin", "pajak", "pendidikan", "konsumen", "ekonomi", "pemerintah", "corona"]

for kw in candidate_keywords:
    count = df["cleaned_content"].str.contains(kw, case=False, na=False).sum()
    print(f"'{kw}': {count} dokumen")

'covid': 6469 dokumen
'vaksin': 2318 dokumen
'pajak': 323 dokumen
'pendidikan': 720 dokumen
'konsumen': 336 dokumen
'ekonomi': 1913 dokumen
'pemerintah': 4972 dokumen
'corona': 1457 dokumen


In [15]:
# Final 5 queries
queries = [
    "kasus covid 19",
    "vaksinasi covid",
    "kebijakan pemerintah",
    "dampak ekonomi",
    "kasus pajak"
]

# Build Retrieval Models

### TF Representation

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

# Reconstruct cleaned/tokenized text as space-joined strings (sklearn expects strings, not token lists)
df["joined_tokens"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

# Build the tf matrix using the vocab we already created
vectorizer_tf = CountVectorizer(vocabulary=vocab)
tf_matrix = vectorizer_tf.fit_transform(df["joined_tokens"])

print("TF matrix shape (documents x vocab terms):", tf_matrix.shape)

TF matrix shape (documents x vocab terms): (14334, 90462)


In [17]:
def preprocess_query(query):
    query = query.lower()
    query = re.sub(r"[^a-zA-Z0-9\s]", " ", query)
    return " ".join(query.split())

def tf_search(query, top_k=10):
    start = time.time()
    
    query_clean = preprocess_query(query)
    query_vec = vectorizer_tf.transform([query_clean])
    
    scores = cosine_similarity(query_vec, tf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

# Example usage
results, elapsed = tf_search("economy stock market")
print(f"TF search time: {elapsed:.4f} seconds")
print(results.head(10))

TF search time: 0.0669 seconds
                                                content     score
8750    Bank Central Asia atau BCA memutuskan untuk ...  0.138675
7625    Wakil Presiden (Wapres), Ma'ruf Amin mengata...  0.093082
8751    Harga saham PT Bank Central Asia Tbk (BBCA) ...  0.090691
6270    Global Islamic Report 2020 mencatat perkemba...  0.080582
3570  PT Garudafood Putra Putri Jaya Tbk (GOOD) meng...  0.077870
864    Pada masa pandemi COVID-19, digitalisasi menj...  0.074019
668     Menteri Badan Usaha Milik Negara (BUMN) Eric...  0.064730
7626    Wakil Presiden (Wapres), Ma'ruf Amin menyebu...  0.051743
4030   Memasak menjadi hobi yang sangat diganrungi s...  0.049281
3512  PT Mandiri Sekuritas (Mansek) mencatat kenaika...  0.047378


In [18]:
# Run all 5 query in model TF
tf_results = {}
tf_times = {}

for q in queries:
    results, elapsed = tf_search(q, top_k=10)
    tf_results[q] = results
    tf_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0565 detik ===


,content,score
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,0.883596
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,0.883022
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,0.883001
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,0.882281
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,0.881119
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,0.868790
2454,Pemerintah melaporkan terdapat 34.379 kasus ba...,0.783841
11940,Kasus positif Covid-19 di tanah air kembali ...,0.783832
2357,Pemerintah melaporkan terdapat 49.071 kasus ba...,0.777444
2459,Pemerintah melaporkan terdapat 31.189 kasus b...,0.777140



=== Query: 'vaksinasi covid' | waktu: 0.0547 detik ===


,content,score
8108,Warga DKI Jakarta bisa melakukan pendaftaran...,0.531975
2891,Ribuan pelajar dari berbagai SMP dan SMA di DK...,0.482742
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,0.480166
4817,UGM tengah mendata para pegawainya untuk diaju...,0.475164
7726,"Wakil Presiden Ma'ruf Amin mengatakan, vaksi...",0.463988
1519,Markas Komando Daerah Militer (Makodam) V/Braw...,0.460287
1415,Program vaksinasi Gotong Royong (VGR) Individu...,0.459901
1904,Pemerintah Daerah (Pemda) Provinsi Jawa Barat...,0.456738
14062,Kementerian Kesehatan mengizinkan penggunaan...,0.456351
12607,Relawan Arus Bawah Jokowi (ABJ) mendukung se...,0.454771



=== Query: 'kebijakan pemerintah' | waktu: 0.0687 detik ===


,content,score
5997,Badan Kepegawaian Negara (BKN) menyampaikan ...,0.457829
7631,Badan Kepegawaian Negara (BKN) melaporkan ju...,0.453692
7642,Badan Kepegawaian Negara (BKN) mencatat juml...,0.452988
7616,Badan Kepegawaian Negara (BKN) mencatat juml...,0.442963
7609,Badan Kepegawaian Negara (BKN) mencatat juml...,0.433577
6015,Badan Kepegawaian Negara (BKN) mencatat seba...,0.429605
7581,Badan Kepegawaian Negara (BKN) mencatat juml...,0.420622
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,0.418237
7579,Badan Kepegawaian Negara (BKN) mencatat juml...,0.412826
7568,Badan Kepegawaian Negara (BKN) mencatat juml...,0.405826



=== Query: 'dampak ekonomi' | waktu: 0.0450 detik ===


,content,score
6074,Menteri Keuangan Sri Mulyani Indrawati menga...,0.340373
6141,Menteri Kesehatan Budi Gunadi Sadikin menila...,0.314786
6075,Wakil Presiden Ma'ruf Amin menghadiri launch...,0.281893
11851,"Presiden Joko Widodo atau Jokowi mengatakan,...",0.272798
5974,Presiden Joko Widodo (Jokowi) memperkirakan ...,0.264039
1071,Pertumbuhan ekonomi Indonesia diproyeksi melam...,0.264039
2703,Bank Indonesia (BI) memprediksi pertumbuhan ek...,0.256424
6111,Karantina wilayah secara total atau lockdown...,0.253917
5985,"Menteri Pariwisata dan Ekonomi Kreatif, Sand...",0.250000
6206,"Deputi Gubernur Bank Indonesia (BI), Doni Pr...",0.242393



=== Query: 'kasus pajak' | waktu: 0.0666 detik ===


,content,score
13995,Penambahan kasus positif Covid-19 di Indones...,0.579480
14033,Kasus positif Covid-19 di Indonesia mengalam...,0.579480
14024,Kasus positif Covid-19 di Indonesia mengalam...,0.578096
13944,Kasus terkonfirmasi positif Covid-19 mengala...,0.575924
13881,Kementerian Kesehatan kembali merilis hasil ...,0.568216
13932,Kasus terkonfirmasi positif Covid-19 di Indo...,0.563771
13215,Penularan Covid-19 semakin masif di Kalimant...,0.562772
4447,Pengamat Pajak Danny Darussalam Tax Center (DD...,0.547120
13923,Kasus terkonfirmasi positif Covid-19 di Indo...,0.543086
13913,Kasus terkonfirmasi positif Covid-19 di Indo...,0.531162


### TF-IDF Representation

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build the tf-idf matrix using the same vocab we already established
vectorizer_tfidf = TfidfVectorizer(vocabulary=vocab)
tfidf_matrix = vectorizer_tfidf.fit_transform(df["joined_tokens"])

print("TF-IDF matrix shape (documents x vocab terms):", tfidf_matrix.shape)

TF-IDF matrix shape (documents x vocab terms): (14334, 90462)


In [20]:
def tfidf_search(query, top_k=10):
    start = time.time()
    
    query_clean = preprocess_query(query)
    query_vec = vectorizer_tfidf.transform([query_clean])
    
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

In [21]:
tfidf_results = {}
tfidf_times = {}

for q in queries:
    results, elapsed = tfidf_search(q, top_k=10)
    tfidf_results[q] = results
    tfidf_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0300 detik ===


,content,score
13913,Kasus terkonfirmasi positif Covid-19 di Indo...,0.685914
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,0.673825
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,0.672682
13923,Kasus terkonfirmasi positif Covid-19 di Indo...,0.671805
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,0.665168
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,0.664157
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,0.662891
13932,Kasus terkonfirmasi positif Covid-19 di Indo...,0.657214
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,0.652731
13944,Kasus terkonfirmasi positif Covid-19 mengala...,0.634597



=== Query: 'vaksinasi covid' | waktu: 0.0395 detik ===


,content,score
7726,"Wakil Presiden Ma'ruf Amin mengatakan, vaksi...",0.492547
8030,Juru Bicara Vaksinasi Covid-19 Kementerian K...,0.469398
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,0.467713
14062,Kementerian Kesehatan mengizinkan penggunaan...,0.461905
8108,Warga DKI Jakarta bisa melakukan pendaftaran...,0.455594
1415,Program vaksinasi Gotong Royong (VGR) Individu...,0.440642
8060,"Pemerintah telah berhasil mencapai target 1,...",0.426202
1900,"Rencana vaksinasi Gotong Royong individu, ata...",0.420988
9405,Rumah Sakit Umum Pusat (RSUP) M Djamil Padan...,0.416466
12820,Kementerian Kesehatan (Kemenkes) optimis Ind...,0.415936



=== Query: 'kebijakan pemerintah' | waktu: 0.0388 detik ===


,content,score
14147,"Sosiolog dari Universitas Gadjah Mada (UGM),...",0.382883
13930,Pandemi Covid-19 masih melanda banyak negara...,0.358856
10897,Pemerintah akan menerapkan kebijakan Pemberl...,0.349760
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,0.339103
6348,Presiden Joko Widodo (Jokowi) menetapkan Pem...,0.328293
11960,Anggota Komisi IX DPR Fraksi PAN Saleh Daula...,0.328163
5315,Ketua Fraksi Partai Amanat Nasional (PAN) DP...,0.325780
5179,Pemerintah belum juga memutuskan kebijakan P...,0.318083
12750,"Anggota Komisi VI DPR RI, Deddy Sitorus meng...",0.315336
2702,Pemerintah mengganti istilah PPKM Darurat menj...,0.303446



=== Query: 'dampak ekonomi' | waktu: 0.0475 detik ===


,content,score
6074,Menteri Keuangan Sri Mulyani Indrawati menga...,0.297221
6141,Menteri Kesehatan Budi Gunadi Sadikin menila...,0.292180
5974,Presiden Joko Widodo (Jokowi) memperkirakan ...,0.245341
11851,"Presiden Joko Widodo atau Jokowi mengatakan,...",0.238470
6075,Wakil Presiden Ma'ruf Amin menghadiri launch...,0.230157
7600,Pemerintah tengah melakukan evaluasi terkait...,0.225548
5985,"Menteri Pariwisata dan Ekonomi Kreatif, Sand...",0.223948
2703,Bank Indonesia (BI) memprediksi pertumbuhan ek...,0.219819
7570,Kamar Dagang dan Industri (Kadin) Indonesia ...,0.211466
1071,Pertumbuhan ekonomi Indonesia diproyeksi melam...,0.201356



=== Query: 'kasus pajak' | waktu: 0.0426 detik ===


,content,score
4447,Pengamat Pajak Danny Darussalam Tax Center (DD...,0.761227
4449,"Bertepatan dengan momentum Hari Pajak 2021, Di...",0.703788
11288,"Sekretaris Daerah (Sekda) Provinsi Bali, Dew...",0.628627
6228,Pemerintah Jokowi memperluas pemajakan melal...,0.538694
1817,Tim penyidik Komisi Pemberatasan Korupsi (KPK)...,0.450886
1634,"Selama bertahun-tahun, laporan pajak dari kepe...",0.441464
6117,"Pemerintah Kabupaten (Pemkab) Karawang, Jawa...",0.430635
6085,Penelitian Big Data Continuum Indonesia mela...,0.426520
14256,"Wakil Ketua Fraksi NasDem DPR RI, Willy Adit...",0.425077
10123,"Wakil Ketua MPR RI Arsul Sani, ikut berpenda...",0.420917


### Word2Vec (Pretrained)

In [23]:
import gensim.downloader as api
from gensim.models import KeyedVectors
import urllib.request
import gzip
import shutil
import os

# URL resmi FastText pretrained vectors untuk bahasa Indonesia
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.id.300.vec.gz"
gz_path = "cc.id.300.vec.gz"
vec_path = "cc.id.300.vec"

# Download (hanya sekali, cukup besar ~1.3GB, jadi mungkin butuh waktu)
if not os.path.exists(vec_path):
    print("Downloading pretrained FastText Indonesian vectors...")
    urllib.request.urlretrieve(url, gz_path)
    
    print("Extracting...")
    with gzip.open(gz_path, "rb") as f_in:
        with open(vec_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

# Load hanya sebagian vocab teratas untuk hemat memori (misal 200,000 kata paling umum)
print("Loading model...")
word2vec_model = KeyedVectors.load_word2vec_format(vec_path, limit=200000)

print("Model loaded. Vector size:", word2vec_model.vector_size)

Extracting...
Loading model...
Model loaded. Vector size: 300


In [24]:
def get_avg_vector(tokens, model):
    """Rata-ratakan vector semua kata dalam dokumen yang ada di vocab model."""
    vectors = [model[word] for word in tokens if word in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)  # dokumen tanpa kata yang dikenali model
    return np.mean(vectors, axis=0)

# Build matrix representasi seluruh dokumen (ini bisa agak lama tergantung jumlah dokumen)
print("Building document vectors...")
doc_vectors = np.array([get_avg_vector(tokens, word2vec_model) for tokens in df["tokens"]])

print("Document vectors shape:", doc_vectors.shape)

Building document vectors...
Document vectors shape: (14334, 300)


In [25]:
from sklearn.metrics.pairwise import cosine_similarity

def word2vec_search(query, top_k=10):
    start = time.time()
    
    query_clean = preprocess_query(query)
    query_tokens = query_clean.split()
    query_vec = get_avg_vector(query_tokens, word2vec_model).reshape(1, -1)
    
    scores = cosine_similarity(query_vec, doc_vectors).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

In [26]:
word2vec_results = {}
word2vec_times = {}

for q in queries:
    results, elapsed = word2vec_search(q, top_k=10)
    word2vec_results[q] = results
    word2vec_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0188 detik ===


,content,score
2459,Pemerintah melaporkan terdapat 31.189 kasus b...,0.920164
2357,Pemerintah melaporkan terdapat 49.071 kasus ba...,0.919847
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,0.916674
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,0.915735
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,0.910638
2454,Pemerintah melaporkan terdapat 34.379 kasus ba...,0.909060
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,0.908886
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,0.908879
11940,Kasus positif Covid-19 di tanah air kembali ...,0.894208
1804,DKI Jakarta kembali mencetak rekor baru penamb...,0.880810



=== Query: 'vaksinasi covid' | waktu: 0.0125 detik ===


,content,score
8755,Indonesia kembali menerima kedatangan vaksin...,0.460490
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,0.456520
12910,Keterlambatan vaksinasi dosis kedua telah te...,0.452622
12391,Kementerian Kesehatan Singapura tidak memasu...,0.450680
1900,"Rencana vaksinasi Gotong Royong individu, ata...",0.444258
11849,Presiden Joko Widodo (Jokowi) menyebut bahwa...,0.442684
2395,Indonesia kembali menerima 1.408.000 dosis vak...,0.442128
12929,"Setelah sempat tidak diperbolehkan, akhirnya...",0.441039
13844,Juru bicara vaksinasi Covid-19 dari Kementer...,0.439481
12801,Pemerintah terus berupaya menjaga pasokan va...,0.436060



=== Query: 'kebijakan pemerintah' | waktu: 0.0101 detik ===


,content,score
7755,"Ketua Komisi XI DPR RI, Dito Ganinduto menga...",0.702016
4389,Otoritas Jasa Keuangan (OJK) memiliki 6 kebij...,0.682801
6090,Posisi aparatur sipil negara (ASN) atau PNS ...,0.680658
6307,Asosiasi Pemerintah Kota Seluruh Indonesia (...,0.678584
4728,Menyikapi eskalasi tindak kekerasan yang terja...,0.677018
10925,Kepala Kepolisian Republik Indonesia (Kapolr...,0.673870
7790,"Wakil Ketua Badan Anggaran (Banggar) DPR RI,...",0.673585
4729,Ketua Gugus Tugas Papua Universitas Gadjah Mad...,0.672895
7653,Anggota Pansus Rancangan Undang-Undang Otono...,0.671840
12747,"Anggota Komisi VI DPR RI dari Fraksi PDIP, D...",0.671646



=== Query: 'dampak ekonomi' | waktu: 0.0113 detik ===


,content,score
364,Faktor penyebab perubahan sosial terjadi dala...,0.695639
4389,Otoritas Jasa Keuangan (OJK) memiliki 6 kebij...,0.693593
13616,Dewan Direktur Eksekutif Bank Dunia menyetuj...,0.691572
7830,Kementerian Perencanaan Pembangunan Nasional...,0.688458
7827,Dewan Direktur Eksekutif Bank Dunia menyetuj...,0.685289
467,Faktor penyebab korupsi bisa disebabkan oleh ...,0.685266
7760,Ketua Komisi XI Dewan Perwakilan Rakyat (DPR...,0.684462
7832,Badan Anggaran DPR RI meminta kepada pemerin...,0.683683
7818,Bank Dunia kembali merilis laporan terbaru m...,0.683280
8119,"Wakil Ketua Badan Anggaran DPR RI, Edhie Bas...",0.682007



=== Query: 'kasus pajak' | waktu: 0.0120 detik ===


,content,score
4447,Pengamat Pajak Danny Darussalam Tax Center (DD...,0.715291
10123,"Wakil Ketua MPR RI Arsul Sani, ikut berpenda...",0.705287
8500,Bea Cukai Bogor kembali bongkar upaya penyel...,0.700812
6085,Penelitian Big Data Continuum Indonesia mela...,0.698127
8907,Pemerintah berencana memungut pajak karbon d...,0.692373
8701,Bea Cukai terus menggalakkan sosialisasi kep...,0.692069
11288,"Sekretaris Daerah (Sekda) Provinsi Bali, Dew...",0.689086
5764,Aliansi Jurnalis Independen (AJI) Kupang ber...,0.687157
5995,Kepala Badan Kebijakan Fiskal (BKF) Kementer...,0.686547
13541,Ketua Umum Asosiasi Pengusaha Indonesia (Api...,0.686036


### Query Likelihood LM (no smoothing)

In [27]:
# Precompute: total kata per dokumen, dan frekuensi tiap kata per dokumen
doc_term_counts = [Counter(tokens) for tokens in df["tokens"]]
doc_lengths = [len(tokens) for tokens in df["tokens"]]

def query_likelihood_basic(query, top_k=10):
    start = time.time()
    
    query_tokens = preprocess_query(query).split()
    
    scores = []
    for i in range(len(df)):
        term_counts = doc_term_counts[i]
        doc_len = doc_lengths[i]
        
        if doc_len == 0:
            scores.append(0.0)
            continue
        
        prob = 1.0
        for w in query_tokens:
            p_w_d = term_counts.get(w, 0) / doc_len
            prob *= p_w_d
        
        scores.append(prob)
    
    scores = np.array(scores)
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

In [28]:
ql_basic_results = {}
ql_basic_times = {}

for q in queries:
    results, elapsed = query_likelihood_basic(q, top_k=10)
    ql_basic_results[q] = results
    ql_basic_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0158 detik ===


,content,score
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,0.001283
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,0.001222
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,0.001222
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,0.001156
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,0.001130
2459,Pemerintah melaporkan terdapat 31.189 kasus b...,0.000852
2357,Pemerintah melaporkan terdapat 49.071 kasus ba...,0.000780
2454,Pemerintah melaporkan terdapat 34.379 kasus ba...,0.000731
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,0.000507
11940,Kasus positif Covid-19 di tanah air kembali ...,0.000424



=== Query: 'vaksinasi covid' | waktu: 0.0024 detik ===


,content,score
8108,Warga DKI Jakarta bisa melakukan pendaftaran...,0.001948
4817,UGM tengah mendata para pegawainya untuk diaju...,0.001514
11474,Gubernur DKI Jakarta Anies Baswedan mengaku ...,0.001357
14064,Pemerintah mengizinkan warga negara asing (W...,0.001258
14062,Kementerian Kesehatan mengizinkan penggunaan...,0.001235
1904,Pemerintah Daerah (Pemda) Provinsi Jawa Barat...,0.001142
12924,Peningkatan minat masyarakat dalam menerima ...,0.001129
9405,Rumah Sakit Umum Pusat (RSUP) M Djamil Padan...,0.001120
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,0.001115
1909,Menteri Kesehatan (Menkes) Budi Gunadi Sadiki...,0.001112



=== Query: 'kebijakan pemerintah' | waktu: 0.0108 detik ===


,content,score
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,0.000774
4791,Peneliti Pusat Studi Ekonomi Kerakyatan (Puste...,0.000730
5179,Pemerintah belum juga memutuskan kebijakan P...,0.000716
6347,"Ketua Asosiasi UMKM Indonesia (Akumindo), Ik...",0.000651
5315,Ketua Fraksi Partai Amanat Nasional (PAN) DP...,0.000599
6348,Presiden Joko Widodo (Jokowi) menetapkan Pem...,0.000584
14147,"Sosiolog dari Universitas Gadjah Mada (UGM),...",0.000583
6392,"Gubernur Bank Indonesia (BI), Perry Warjiyo ...",0.000558
5199,Menko Kemaritiman dan Investasi Luhut Binsar...,0.000540
2702,Pemerintah mengganti istilah PPKM Darurat menj...,0.000515



=== Query: 'dampak ekonomi' | waktu: 0.0119 detik ===


,content,score
5985,"Menteri Pariwisata dan Ekonomi Kreatif, Sand...",0.000336
7570,Kamar Dagang dan Industri (Kadin) Indonesia ...,0.000185
5293,Sejumlah sekretaris jenderal dan politikus k...,0.000142
7829,"Anggota Banggar DPR RI, Eko Hendro Purnomo a...",0.000140
7575,Ketua Umum Kamar Dagang Industri (Kadin) Ind...,0.000137
407,Pemberlakuan pembatasan kegiatan masyarakat (...,0.000134
5909,"Menteri Koordinator bidang Perekonomian, Air...",0.000134
7600,Pemerintah tengah melakukan evaluasi terkait...,0.000132
5973,Menteri Koordinator bidang Perekonomian Airl...,0.000128
6141,Menteri Kesehatan Budi Gunadi Sadikin menila...,0.000120



=== Query: 'kasus pajak' | waktu: 0.0108 detik ===


,content,score
1817,Tim penyidik Komisi Pemberatasan Korupsi (KPK)...,0.000481
1747,Mantan Direktur Pemeriksaan dan Penagihan pad...,0.000347
7765,"Kementerian Keuangan mencatat, jumlah wajib ...",0.000235
1746,Komisi Pemberantasan Korupsi (KPK) menyebut t...,0.000195
11419,Hakim Tunggal Pengadilan Negeri Jakarta Sela...,0.000188
11398,Penyidik Pegawai Negeri Sipil (PPNS) Kantor ...,0.000158
4449,"Bertepatan dengan momentum Hari Pajak 2021, Di...",0.000153
530,Pemerintah telah menjalankan Pemberlakuan Pemb...,0.000121
11961,Pemerintah berencana menerapkan Pemberlakuan...,0.000095
6145,Menteri Keuangan Sri Mulyani Indrawati menca...,0.000083


In [29]:
def zero_score_analysis(query):
    query_tokens = preprocess_query(query).split()
    
    zero_count = 0
    total_docs = len(df)
    
    for i in range(total_docs):
        term_counts = doc_term_counts[i]
        doc_len = doc_lengths[i]
        
        if doc_len == 0:
            zero_count += 1
            continue
        
        prob = 1.0
        for w in query_tokens:
            p_w_d = term_counts.get(w, 0) / doc_len
            prob *= p_w_d
        
        if prob == 0.0:
            zero_count += 1
    
    pct_zero = (zero_count / total_docs) * 100
    return zero_count, total_docs, pct_zero

# Jalankan untuk semua 5 query
print(f"{'Query':<25} {'Zero-score docs':<20} {'Total docs':<12} {'% Zero'}")
print("-" * 75)
for q in queries:
    zero_count, total_docs, pct_zero = zero_score_analysis(q)
    print(f"{q:<25} {zero_count:<20} {total_docs:<12} {pct_zero:.2f}%")

Query                     Zero-score docs      Total docs   % Zero
---------------------------------------------------------------------------
kasus covid 19            11694                14334        81.58%
vaksinasi covid           12802                14334        89.31%
kebijakan pemerintah      13140                14334        91.67%
dampak ekonomi            14056                14334        98.06%
kasus pajak               14280                14334        99.62%


### Query Likelihood (Add-One/Laplace Smoothing)

In [30]:
V = len(vocab)  # ukuran vocabulary, dibutuhkan untuk normalisasi Laplace

def query_likelihood_laplace(query, top_k=10):
    start = time.time()
    
    query_tokens = preprocess_query(query).split()
    
    scores = []
    for i in range(len(df)):
        term_counts = doc_term_counts[i]
        doc_len = doc_lengths[i]
        
        log_prob = 0.0
        for w in query_tokens:
            count_w_d = term_counts.get(w, 0)
            p_w_d = (count_w_d + 1) / (doc_len + V)
            log_prob += np.log(p_w_d)
        
        scores.append(log_prob)
    
    scores = np.array(scores)
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

In [31]:
ql_laplace_results = {}
ql_laplace_times = {}

for q in queries:
    results, elapsed = query_likelihood_laplace(q, top_k=10)
    ql_laplace_results[q] = results
    ql_laplace_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0264 detik ===


,content,score
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,-22.904883
11940,Kasus positif Covid-19 di tanah air kembali ...,-22.927254
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,-22.991123
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,-22.991222
2454,Pemerintah melaporkan terdapat 34.379 kasus ba...,-23.066334
13975,Ada lebih dari 7.100 anak dan remaja meningg...,-23.104864
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,-23.110114
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,-23.134310
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,-23.134310
2357,Pemerintah melaporkan terdapat 49.071 kasus ba...,-23.162072



=== Query: 'vaksinasi covid' | waktu: 0.0225 detik ===


,content,score
1414,Program Vaksinasi Gotong Royong (VGR) Individu...,-15.358039
119,"Pagi-pagi, antrean sudah terlihat di depan Kl...",-15.609871
1415,Program vaksinasi Gotong Royong (VGR) Individu...,-16.187296
12847,Pemerintah memulai program vaksinasi Covid-1...,-16.292743
1379,Lembaga Survei Indonesia (LSI) merilis hasil s...,-16.707724
13553,Pemerintah terus menggencarkan program vaksi...,-16.813543
5228,Anggota Komisi IX DPR Fraksi PKS Kurniasih M...,-16.990373
8873,Pandemi Covid-19 telah merebak lebih dari se...,-17.046168
5232,Anggota DPR Fraksi Demokrat Irwan Fecho meni...,-17.110479
6042,"Beberapa hari terakhir, gelombang penerimaan...",-17.127879



=== Query: 'kebijakan pemerintah' | waktu: 0.0189 detik ===


,content,score
1414,Program Vaksinasi Gotong Royong (VGR) Individu...,-16.469555
1370,Penerapan Pembatasan Kegiatan Masyarakat atau ...,-16.542272
13576,Presiden Joko Widodo (Jokowi) menetapkan Pem...,-16.854125
13930,Pandemi Covid-19 masih melanda banyak negara...,-17.093363
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,-17.438188
10029,Ciri-Ciri demokrasi seharusnya merupakan hal...,-17.524109
13553,Pemerintah terus menggencarkan program vaksi...,-17.606774
12750,"Anggota Komisi VI DPR RI, Deddy Sitorus meng...",-17.715001
6519,Pemerintah memutuskan memperpanjang pelaksan...,-17.850560
10123,"Wakil Ketua MPR RI Arsul Sani, ikut berpenda...",-17.880844



=== Query: 'dampak ekonomi' | waktu: 0.0169 detik ===


,content,score
13576,Presiden Joko Widodo (Jokowi) menetapkan Pem...,-18.766134
6075,Wakil Presiden Ma'ruf Amin menghadiri launch...,-18.896792
14322,"Nama Taman Nasional (TN) Komodo, Manggarai B...",-19.081367
13558,Setelah sempat berada di level atas untuk ne...,-19.111983
1370,Penerapan Pembatasan Kegiatan Masyarakat atau ...,-19.132539
7829,"Anggota Banggar DPR RI, Eko Hendro Purnomo a...",-19.146349
4934,Kisi-kisi potongan batang bambu nampak masih ...,-19.163414
7731,Kamar Dagang dan Industri (Kadin) Indonesia ...,-19.177116
7838,Asisten Deputi Pengembangan Industri Kemente...,-19.255929
14313,Terpilihnya Sandiaga Uno sebagai Menteri Par...,-19.286020



=== Query: 'kasus pajak' | waktu: 0.0121 detik ===


,content,score
4449,"Bertepatan dengan momentum Hari Pajak 2021, Di...",-18.531999
11062,Indonesia Corruption Watch (ICW) memberikan ...,-18.749511
12427,Badan Kesehatan Dunia (WHO) mengatakan varia...,-18.806412
1747,Mantan Direktur Pemeriksaan dan Penagihan pad...,-18.881233
11419,Hakim Tunggal Pengadilan Negeri Jakarta Sela...,-18.883787
1817,Tim penyidik Komisi Pemberatasan Korupsi (KPK)...,-18.959953
1746,Komisi Pemberantasan Korupsi (KPK) menyebut t...,-18.963235
13913,Kasus terkonfirmasi positif Covid-19 di Indo...,-18.983540
13923,Kasus terkonfirmasi positif Covid-19 di Indo...,-19.026321
6439,Satgas Penanganan Covid-19 menyatakan ada ti...,-19.048007


### Query Likelihood (Linear Interpolation Smoothing (Jelinek-Mercer))

In [32]:
# Precompute: model bahasa koleksi P(w|C) untuk SEMUA kata di vocab
collection_length = sum(doc_lengths)  # total kata di seluruh corpus
collection_term_counts = term_freq  # sudah kita hitung di Task 1 (Counter seluruh corpus)

def get_collection_prob(word):
    return collection_term_counts.get(word, 0) / collection_length

lam = 0.7  # bobot untuk model dokumen; (1 - lam) untuk model koleksi

def query_likelihood_interpolation(query, top_k=10, lam=0.7):
    start = time.time()
    
    query_tokens = preprocess_query(query).split()
    
    # Precompute P(w|C) untuk tiap kata query (sama untuk semua dokumen, jadi hitung sekali saja)
    p_w_c = {w: get_collection_prob(w) for w in query_tokens}
    
    scores = []
    for i in range(len(df)):
        term_counts = doc_term_counts[i]
        doc_len = doc_lengths[i]
        
        log_prob = 0.0
        for w in query_tokens:
            p_w_d = term_counts.get(w, 0) / doc_len if doc_len > 0 else 0
            p_mix = lam * p_w_d + (1 - lam) * p_w_c[w]
            
            if p_mix > 0:
                log_prob += np.log(p_mix)
            else:
                log_prob += np.log(1e-12)  # fallback kecil kalau kata sama sekali tidak ada di corpus
        
        scores.append(log_prob)
    
    scores = np.array(scores)
    top_indices = scores.argsort()[::-1][:top_k]
    
    elapsed = time.time() - start
    
    results = df.iloc[top_indices][["content"]].copy()
    results["score"] = scores[top_indices]
    
    return results, elapsed

In [33]:
ql_interp_results = {}
ql_interp_times = {}

for q in queries:
    results, elapsed = query_likelihood_interpolation(q, top_k=10, lam=0.7)
    ql_interp_results[q] = results
    ql_interp_times[q] = elapsed
    print(f"\n=== Query: '{q}' | waktu: {elapsed:.4f} detik ===")
    display(results[["content", "score"]])


=== Query: 'kasus covid 19' | waktu: 0.0343 detik ===


,content,score
13796,Kasus terkonfirmasi positif Covid-19 di Indo...,-7.679218
13825,Kasus terkonfirmasi positif Covid-19 di Indo...,-7.727382
13815,Kasus terkonfirmasi positif Covid-19 di Indo...,-7.727382
13861,Kasus terkonfirmasi positif Covid-19 di Indo...,-7.782060
13848,Kasus terkonfirmasi positif Covid-19 di Indo...,-7.804264
2459,Pemerintah melaporkan terdapat 31.189 kasus b...,-8.080327
2357,Pemerintah melaporkan terdapat 49.071 kasus ba...,-8.167042
2454,Pemerintah melaporkan terdapat 34.379 kasus ba...,-8.230701
1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,-8.590871
11940,Kasus positif Covid-19 di tanah air kembali ...,-8.765167



=== Query: 'vaksinasi covid' | waktu: 0.0266 detik ===


,content,score
8108,Warga DKI Jakarta bisa melakukan pendaftaran...,-6.889555
4817,UGM tengah mendata para pegawainya untuk diaju...,-7.143184
11474,Gubernur DKI Jakarta Anies Baswedan mengaku ...,-7.243075
14064,Pemerintah mengizinkan warga negara asing (W...,-7.300310
14062,Kementerian Kesehatan mengizinkan penggunaan...,-7.322608
1904,Pemerintah Daerah (Pemda) Provinsi Jawa Barat...,-7.390175
2423,Kepala Staf Kepresidenan Moeldoko mengatakan v...,-7.390855
9405,Rumah Sakit Umum Pusat (RSUP) M Djamil Padan...,-7.399940
12924,Peningkatan minat masyarakat dalam menerima ...,-7.411479
1909,Menteri Kesehatan (Menkes) Budi Gunadi Sadiki...,-7.414717



=== Query: 'kebijakan pemerintah' | waktu: 0.0211 detik ===


,content,score
6221,Ketua Asosiasi Pengelola Pusat Belanja Indon...,-7.827463
4791,Peneliti Pusat Studi Ekonomi Kerakyatan (Puste...,-7.890144
5179,Pemerintah belum juga memutuskan kebijakan P...,-7.908101
6347,"Ketua Asosiasi UMKM Indonesia (Akumindo), Ik...",-8.002830
5315,Ketua Fraksi Partai Amanat Nasional (PAN) DP...,-8.076814
14147,"Sosiolog dari Universitas Gadjah Mada (UGM),...",-8.088813
6348,Presiden Joko Widodo (Jokowi) menetapkan Pem...,-8.092774
6392,"Gubernur Bank Indonesia (BI), Perry Warjiyo ...",-8.157879
5199,Menko Kemaritiman dan Investasi Luhut Binsar...,-8.180351
2702,Pemerintah mengganti istilah PPKM Darurat menj...,-8.227711



=== Query: 'dampak ekonomi' | waktu: 0.0214 detik ===


,content,score
5985,"Menteri Pariwisata dan Ekonomi Kreatif, Sand...",-8.688354
7570,Kamar Dagang dan Industri (Kadin) Indonesia ...,-9.282633
5293,Sejumlah sekretaris jenderal dan politikus k...,-9.541590
7829,"Anggota Banggar DPR RI, Eko Hendro Purnomo a...",-9.556182
7575,Ketua Umum Kamar Dagang Industri (Kadin) Ind...,-9.581813
407,Pemberlakuan pembatasan kegiatan masyarakat (...,-9.593572
5909,"Menteri Koordinator bidang Perekonomian, Air...",-9.603835
7600,Pemerintah tengah melakukan evaluasi terkait...,-9.618247
5973,Menteri Koordinator bidang Perekonomian Airl...,-9.645877
6141,Menteri Kesehatan Budi Gunadi Sadikin menila...,-9.701265



=== Query: 'kasus pajak' | waktu: 0.0178 detik ===


,content,score
1817,Tim penyidik Komisi Pemberatasan Korupsi (KPK)...,-8.264571
1747,Mantan Direktur Pemeriksaan dan Penagihan pad...,-8.570852
7765,"Kementerian Keuangan mencatat, jumlah wajib ...",-8.927383
4449,"Bertepatan dengan momentum Hari Pajak 2021, Di...",-9.089552
1746,Komisi Pemberantasan Korupsi (KPK) menyebut t...,-9.117635
11419,Hakim Tunggal Pengadilan Negeri Jakarta Sela...,-9.149823
11398,Penyidik Pegawai Negeri Sipil (PPNS) Kantor ...,-9.294718
530,Pemerintah telah menjalankan Pemberlakuan Pemb...,-9.510722
4447,Pengamat Pajak Danny Darussalam Tax Center (DD...,-9.771441
11209,Penyidik Pegawai Negeri Sipil (PPNS) Kantor ...,-9.800578


# Testing

In [36]:
# Tabel Perbandingan Waktu Komputasi
all_times = {
    "TF": tf_times,
    "TF-IDF": tfidf_times,
    "Word2Vec": word2vec_times,
    "Basic QL": ql_basic_times,
    "Laplace": ql_laplace_times,
    "Linear Interpolation": ql_interp_times
}

time_df = pd.DataFrame(all_times, index=queries)
time_df.index.name = "Query"

# Tambahkan baris rata-rata di bawah
time_df.loc["Rata-rata"] = time_df.mean()

print("=== Perbandingan Waktu Komputasi (detik) ===")
display(time_df.round(4))

=== Perbandingan Waktu Komputasi (detik) ===


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
Query,,,,,,
kasus covid 19,0.0565,0.0300,0.0188,0.0158,0.0264,0.0343
vaksinasi covid,0.0547,0.0395,0.0125,0.0024,0.0225,0.0266
kebijakan pemerintah,0.0687,0.0388,0.0101,0.0108,0.0189,0.0211
dampak ekonomi,0.0450,0.0475,0.0113,0.0119,0.0169,0.0214
kasus pajak,0.0666,0.0426,0.0120,0.0108,0.0121,0.0178
Rata-rata,0.0583,0.0397,0.0130,0.0103,0.0194,0.0242


In [37]:
# Tabel Perbandingan Top 10 Dokumen (per query, semua model)
all_model_results = {
    "TF": tf_results,
    "TF-IDF": tfidf_results,
    "Word2Vec": word2vec_results,
    "Basic QL": ql_basic_results,
    "Laplace": ql_laplace_results,
    "Linear Interpolation": ql_interp_results
}

def compare_top10_per_query(query):
    print(f"\n{'='*80}\nQUERY: '{query}'\n{'='*80}")
    for model_name, results_dict in all_model_results.items():
        docs = results_dict[query]["content"].str[:80].tolist()  # potong biar rapi
        print(f"\n--- {model_name} ---")
        for rank, doc in enumerate(docs, 1):
            print(f"{rank}. {doc}...")

# Jalankan untuk semua query
for q in queries:
    compare_top10_per_query(q)


QUERY: 'kasus covid 19'

--- TF ---
1.   Kasus terkonfirmasi positif Covid-19 di Indonesia mengalami penambahan sebanya...
2.   Kasus terkonfirmasi positif Covid-19 di Indonesia mengalami penambahan 38.325 ...
3.   Kasus terkonfirmasi positif Covid-19 di Indonesia mengalami penambahan sebanya...
4.   Kasus terkonfirmasi positif Covid-19 di Indonesia mengalami penambahan sebanya...
5.   Kasus terkonfirmasi positif Covid-19 di Indonesia mengalami penambahan sebanya...
6. PT Wijaya Karya (Persero) Tbk. (WIKA) menyerahkan 1.145 Alat Pelindung Diri (APD...
7. Pemerintah melaporkan terdapat 34.379 kasus baru Covid-19 di Indonesia, Rabu (7/...
8.   Kasus positif Covid-19 di tanah air kembali melonjak tinggi. Dari data Satgas ...
9. Pemerintah melaporkan terdapat 49.071 kasus baru virus corona (Covid-19) di Indo...
10.  Pemerintah melaporkan terdapat 31.189 kasus baru virus corona (Covid-19) di Ind...

--- TF-IDF ---
1.   Kasus terkonfirmasi positif Covid-19 di Indonesia mengalami penambahan 

In [38]:
# Analisis Overlap Dokumen Antar Model (kuantitatif)
all_model_indices = {
    "TF": tf_results,
    "TF-IDF": tfidf_results,
    "Word2Vec": word2vec_results,
    "Basic QL": ql_basic_results,
    "Laplace": ql_laplace_results,
    "Linear Interpolation": ql_interp_results
}

def overlap_matrix(query):
    model_names = list(all_model_indices.keys())
    n = len(model_names)
    overlap = pd.DataFrame(index=model_names, columns=model_names, dtype=int)
    
    doc_sets = {name: set(all_model_indices[name][query].index) for name in model_names}
    
    for m1 in model_names:
        for m2 in model_names:
            overlap.loc[m1, m2] = len(doc_sets[m1] & doc_sets[m2])
    
    return overlap

# Contoh untuk satu query dulu
print("Overlap matrix untuk query 'kasus pajak':")
display(overlap_matrix("kasus pajak"))

Overlap matrix untuk query 'kasus pajak':


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
TF,10.0,1.0,1.0,0.0,2.0,1.0
TF-IDF,1.0,10.0,4.0,2.0,2.0,3.0
Word2Vec,1.0,4.0,10.0,0.0,0.0,1.0
Basic QL,0.0,2.0,0.0,10.0,5.0,8.0
Laplace,2.0,2.0,0.0,5.0,10.0,5.0
Linear Interpolation,1.0,3.0,1.0,8.0,5.0,10.0


In [40]:
# Overlap matrix tiap query dalam dictionary
overlap_matrices = {}

for q in queries:
    overlap_matrices[q] = overlap_matrix(q)
    print(f"\n{'='*60}")
    print(f"Overlap matrix - Query: '{q}'")
    print(f"{'='*60}")
    display(overlap_matrices[q])


Overlap matrix - Query: 'kasus covid 19'


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
TF,10.0,6.0,9.0,10.0,9.0,10.0
TF-IDF,6.0,10.0,5.0,6.0,6.0,6.0
Word2Vec,9.0,5.0,10.0,9.0,8.0,9.0
Basic QL,10.0,6.0,9.0,10.0,9.0,10.0
Laplace,9.0,6.0,8.0,9.0,10.0,9.0
Linear Interpolation,10.0,6.0,9.0,10.0,9.0,10.0



Overlap matrix - Query: 'vaksinasi covid'


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
TF,10.0,5.0,1.0,5.0,1.0,5.0
TF-IDF,5.0,10.0,2.0,4.0,1.0,4.0
Word2Vec,1.0,2.0,10.0,1.0,0.0,1.0
Basic QL,5.0,4.0,1.0,10.0,0.0,10.0
Laplace,1.0,1.0,0.0,0.0,10.0,0.0
Linear Interpolation,5.0,4.0,1.0,10.0,0.0,10.0



Overlap matrix - Query: 'kebijakan pemerintah'


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
TF,10.0,1.0,0.0,1.0,1.0,1.0
TF-IDF,1.0,10.0,0.0,6.0,3.0,6.0
Word2Vec,0.0,0.0,10.0,0.0,0.0,0.0
Basic QL,1.0,6.0,0.0,10.0,1.0,10.0
Laplace,1.0,3.0,0.0,1.0,10.0,1.0
Linear Interpolation,1.0,6.0,0.0,10.0,1.0,10.0



Overlap matrix - Query: 'dampak ekonomi'


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
TF,10.0,8.0,0.0,2.0,1.0,2.0
TF-IDF,8.0,10.0,0.0,4.0,1.0,4.0
Word2Vec,0.0,0.0,10.0,0.0,0.0,0.0
Basic QL,2.0,4.0,0.0,10.0,1.0,10.0
Laplace,1.0,1.0,0.0,1.0,10.0,1.0
Linear Interpolation,2.0,4.0,0.0,10.0,1.0,10.0



Overlap matrix - Query: 'kasus pajak'


,TF,TF-IDF,Word2Vec,Basic QL,Laplace,Linear Interpolation
TF,10.0,1.0,1.0,0.0,2.0,1.0
TF-IDF,1.0,10.0,4.0,2.0,2.0,3.0
Word2Vec,1.0,4.0,10.0,0.0,0.0,1.0
Basic QL,0.0,2.0,0.0,10.0,5.0,8.0
Laplace,2.0,2.0,0.0,5.0,10.0,5.0
Linear Interpolation,1.0,3.0,1.0,8.0,5.0,10.0


# Evaluation

In [47]:
# Pooling —> Buat Candidate Set untuk Ground Truth

def build_pooling_set(query, all_model_results, top_k=10):
    """Union semua doc_id (index DataFrame) yang muncul di top-k semua model untuk 1 query."""
    pooled_ids = set()
    for model_name, results_per_query in all_model_results.items():
        top_docs_df = results_per_query[query].head(top_k)
        pooled_ids.update(top_docs_df.index.tolist())
    return pooled_ids

pooling_sets = {q: build_pooling_set(q, all_model_results, top_k=10) for q in queries}

for q, ids in pooling_sets.items():
    print(f"Query: '{q}' -> {len(ids)} dokumen unik perlu di-judge")

Query: 'kasus covid 19' -> 16 dokumen unik perlu di-judge
Query: 'vaksinasi covid' -> 36 dokumen unik perlu di-judge
Query: 'kebijakan pemerintah' -> 40 dokumen unik perlu di-judge
Query: 'dampak ekonomi' -> 36 dokumen unik perlu di-judge
Query: 'kasus pajak' -> 37 dokumen unik perlu di-judge


In [ ]:
# export ke excel
import openpyxl

judge_rows = []
for q, doc_ids in pooling_sets.items():
    for doc_id in doc_ids:
        content_snippet = df.loc[doc_id, "content"][:300]
        judge_rows.append({
            "query": q,
            "doc_id": doc_id,
            "content_snippet": content_snippet,
            "relevance": ""
        })

judge_df = pd.DataFrame(judge_rows)
judge_df.to_excel("relevance_ground_truth.xlsx", index=False)
print(f"Total baris untuk di-judge: {len(judge_df)}")
judge_df.head()

Total baris untuk di-judge: 165


,query,doc_id,content_snippet,relevance
0,kasus covid 19,13825,Kasus terkonfirmasi positif Covid-19 di Indo...,
1,kasus covid 19,1090,PT Wijaya Karya (Persero) Tbk. (WIKA) menyerah...,
2,kasus covid 19,13923,Kasus terkonfirmasi positif Covid-19 di Indo...,
3,kasus covid 19,13796,Kasus terkonfirmasi positif Covid-19 di Indo...,
4,kasus covid 19,13861,Kasus terkonfirmasi positif Covid-19 di Indo...,


In [63]:
judge_df = pd.read_excel("relevance_ground_truth.xlsx")  # yang udah kamu isi

# Convert jadi ground_truth dict: {query: {doc_id: relevance_score}}
ground_truth = {}
for q in queries:
    sub = judge_df[judge_df["query"] == q]
    ground_truth[q] = dict(zip(sub["doc_id"], sub["relevance"]))

In [64]:
def precision_at_k(retrieved_ids, relevant_dict, k=10):
    top_k = retrieved_ids[:k]
    relevant_count = sum(1 for doc_id in top_k if relevant_dict.get(doc_id, 0) > 0)
    return relevant_count / k

def recall_at_k(retrieved_ids, relevant_dict, k=10):
    top_k = retrieved_ids[:k]
    total_relevant = sum(1 for v in relevant_dict.values() if v > 0)
    if total_relevant == 0:
        return 0.0
    relevant_count = sum(1 for doc_id in top_k if relevant_dict.get(doc_id, 0) > 0)
    return relevant_count / total_relevant

def average_precision(retrieved_ids, relevant_dict, k=10):
    top_k = retrieved_ids[:k]
    hits = 0
    sum_precisions = 0.0
    for i, doc_id in enumerate(top_k, start=1):
        if relevant_dict.get(doc_id, 0) > 0:
            hits += 1
            sum_precisions += hits / i
    if hits == 0:
        return 0.0
    return sum_precisions / hits

def dcg_at_k(retrieved_ids, relevant_dict, k=10):
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k], start=1):
        rel = relevant_dict.get(doc_id, 0)
        dcg += (2**rel - 1) / np.log2(i + 1)
    return dcg

def ndcg_at_k(retrieved_ids, relevant_dict, k=10):
    dcg = dcg_at_k(retrieved_ids, relevant_dict, k)
    ideal_order = sorted(relevant_dict.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / np.log2(i + 1) for i, rel in enumerate(ideal_order, start=1))
    if idcg == 0:
        return 0.0
    return dcg / idcg

def reciprocal_rank(retrieved_ids, relevant_dict, k=10):
    for i, doc_id in enumerate(retrieved_ids[:k], start=1):
        if relevant_dict.get(doc_id, 0) > 0:
            return 1 / i
    return 0.0

In [65]:
def evaluate_model(model_results, ground_truth, queries, k=10):
    """model_results: {query: DataFrame dengan index=doc_id, kolom=['content','score']}"""
    metrics_per_query = []
    for q in queries:
        retrieved_ids = model_results[q].index.tolist()  # <-- bukan unpacking tuple
        relevant_dict = ground_truth[q]

        metrics_per_query.append({
            "query": q,
            "precision@10": precision_at_k(retrieved_ids, relevant_dict, k),
            "recall@10": recall_at_k(retrieved_ids, relevant_dict, k),
            "AP": average_precision(retrieved_ids, relevant_dict, k),
            "nDCG@10": ndcg_at_k(retrieved_ids, relevant_dict, k),
            "RR": reciprocal_rank(retrieved_ids, relevant_dict, k),
        })
    return pd.DataFrame(metrics_per_query)

evaluation_results = {}
for model_name, results in all_model_results.items():
    evaluation_results[model_name] = evaluate_model(results, ground_truth, queries, k=10)

In [66]:
summary_rows = []
for model_name, metrics_df in evaluation_results.items():
    summary_rows.append({
        "Model": model_name,
        "Precision@10": metrics_df["precision@10"].mean(),
        "Recall@10": metrics_df["recall@10"].mean(),
        "MAP": metrics_df["AP"].mean(),
        "nDCG@10": metrics_df["nDCG@10"].mean(),
        "MRR": metrics_df["RR"].mean(),
    })

final_comparison_df = pd.DataFrame(summary_rows).set_index("Model")
final_comparison_df = final_comparison_df.round(4)

print("=== Perbandingan Akhir Semua Model ===")
final_comparison_df.sort_values("MAP", ascending=False)

=== Perbandingan Akhir Semua Model ===


,Precision@10,Recall@10,MAP,nDCG@10,MRR
Model,,,,,
Basic QL,0.84,0.4032,0.9526,0.7748,1.00
Linear Interpolation,0.88,0.4298,0.9479,0.7989,1.00
Word2Vec,0.84,0.3970,0.8923,0.8040,0.90
TF-IDF,0.84,0.4032,0.8709,0.6255,0.90
Laplace,0.68,0.3194,0.6916,0.4807,0.65
TF,0.56,0.2660,0.5636,0.4733,0.55


Best Performing Model --> Query Likelihood with Linear Interpolation <br>
Linear Interpolation paling seimbang di semua metrik (Precision 0.88 tertinggi, MAP 0.9479 dekat Basic QL, MRR 1.00) <br>
ini kandidat paling aman buat direkomendasikan sebagai "overall best" karena konsisten di semua metrik, bukan cuma menang di satu

# Enhancement

In [67]:
!pip install Sastrawi


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
# Stemming + Stopwords Removal
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

def normalize_text(text):
    """Lowercase (sudah dari cleaning sebelumnya) -> stopword removal -> stemming"""
    text_no_stopwords = stopword_remover.remove(text)
    text_stemmed = stemmer.stem(text_no_stopwords)
    return text_stemmed

In [ ]:
from tqdm import tqdm
tqdm.pandas()

df["content_normalized"] = df["content"].progress_apply(normalize_text)
df["tokens_normalized"] = df["content_normalized"].apply(lambda x: x.split())
df["joined_tokens_normalized"] = df["tokens_normalized"].apply(lambda x: " ".join(x))

In [ ]:
from collections import Counter

all_tokens_norm = [t for tokens in df["tokens_normalized"] for t in tokens]
term_freq_norm = Counter(all_tokens_norm)
vocab_norm = sorted(term_freq_norm.keys())

print(f"Vocab sebelum normalisasi: {len(vocab)} term")
print(f"Vocab sesudah normalisasi: {len(vocab_norm)} term")

In [ ]:
doc_term_counts_norm = [Counter(tokens) for tokens in df["tokens_normalized"]]
doc_lengths_norm = [len(tokens) for tokens in df["tokens_normalized"]]

# Collection-level term counts untuk background model P(w|C) di linear interpolation
collection_term_counts_norm = Counter(all_tokens_norm)
collection_length_norm = sum(doc_lengths_norm)

In [ ]:
def preprocess_query_normalized(query):
    query_clean = query.lower()
    return normalize_text(query_clean)

In [ ]:
lambda_param = 0.7  # sesuaikan sama parameter yang kamu pakai sebelumnya

def query_likelihood_linear_interp_normalized(query, top_k=10):
    start = time.time()
    query_tokens = preprocess_query_normalized(query).split()

    scores = []
    for i in range(len(df)):
        term_counts = doc_term_counts_norm[i]
        doc_len = doc_lengths_norm[i]

        log_prob = 0
        for term in query_tokens:
            p_doc = term_counts.get(term, 0) / doc_len if doc_len > 0 else 0
            p_collection = collection_term_counts_norm.get(term, 0) / collection_length_norm
            p_smoothed = lambda_param * p_doc + (1 - lambda_param) * p_collection
            if p_smoothed > 0:
                log_prob += np.log(p_smoothed)
            else:
                log_prob += -np.inf
        scores.append((i, log_prob))

    scores.sort(key=lambda x: x[1], reverse=True)
    top_docs = scores[:top_k]
    result_df = pd.DataFrame(
        [(doc_id, df.loc[doc_id, "content"][:100], score) for doc_id, score in top_docs],
        columns=["doc_id", "content", "score"]
    ).set_index("doc_id")

    elapsed = time.time() - start
    return result_df, elapsed

# Jalankan untuk semua 5 query
interp_norm_results = {}
interp_norm_times = {}
for q in queries:
    results, elapsed = query_likelihood_linear_interp_normalized(q, top_k=10)
    interp_norm_results[q] = results
    interp_norm_times[q] = elapsed
    print(f"Query '{q}' selesai dalam {elapsed:.4f} detik")

In [ ]:
interp_norm_eval = evaluate_model({q: interp_norm_results[q] for q in queries}, ground_truth, queries, k=10)

before_after = pd.DataFrame({
    "Before (Linear Interp)": final_comparison_df.loc["Linear Interpolation"],
    "After (Normalized)": interp_norm_eval[["precision@10","recall@10","AP","nDCG@10","RR"]].mean().rename({
        "precision@10": "Precision@10", "recall@10": "Recall@10",
        "AP": "MAP", "nDCG@10": "nDCG@10", "RR": "MRR"
    })
})
before_after["Δ"] = before_after["After (Normalized)"] - before_after["Before (Linear Interp)"]

print("=== Before vs After Normalization ===")
before_after.round(4)